# 🚦 Notebook 3: Stack Overflow — Real-world extensions

Interviewers love the follow-up *"ok, now what if…?"* This notebook adds five common ones on
top of the *best* version from notebook 2:

1. **Tags + search** — find questions by tag or keyword.
2. **Badges via the observer pattern** — automatic award when a post crosses a threshold.
3. **Moderation** — close / reopen / delete a question.
4. **Thread-safe voting** — two voters hitting the same post at once.
5. **Pluggable reputation rules (strategy pattern)** — different rules for different sites.

Each section is self-contained: you can jump to whichever one you're curious about.


## 🛠️ Setup

```bash
cd 07-object-oriented-design/stack-overflow
uv sync
```

Select the `.venv` kernel. If not visible, reload: `Cmd+Shift+P` → **Reload Window**.

We'll re-declare the core classes here so this notebook is fully self-contained.


In [1]:
from __future__ import annotations
from dataclasses import dataclass, field
from enum import Enum
from typing import Optional, List, Callable
import itertools, threading, time

REP_UPVOTE_QUESTION = 5
REP_UPVOTE_ANSWER   = 10
REP_DOWNVOTE        = -2
REP_ACCEPTED_BONUS  = 15

class VoteType(Enum):
    UP   = +1
    DOWN = -1

class QuestionStatus(Enum):
    OPEN    = 'open'
    CLOSED  = 'closed'
    DELETED = 'deleted'

@dataclass
class User:
    id: str
    name: str
    reputation: int = 0
    badges: list = field(default_factory=list)

_pid = itertools.count(1)

class Post:
    def __init__(self, author: User, body: str):
        self.id = next(_pid)
        self.author = author
        self.body = body
        self.votes = {}
        self.comments = []
        self._listeners = []   # observer hook used in section 2

    @property
    def score(self) -> int:
        return sum(v.value for v in self.votes.values())

    def comment(self, user: User, text: str) -> None:
        self.comments.append((user.id, text))

    def subscribe(self, fn: Callable) -> None:
        self._listeners.append(fn)

    def _notify(self) -> None:
        for fn in self._listeners:
            fn(self)

class Question(Post):
    def __init__(self, author, title, body, tags):
        super().__init__(author, body)
        self.title = title
        self.tags = list(tags)
        self.answers = []
        self.accepted = None
        self.status = QuestionStatus.OPEN

    def _add_answer(self, ans):
        if self.status is not QuestionStatus.OPEN:
            raise ValueError(f'cannot answer a {self.status.value} question')
        self.answers.append(ans)

    def accept(self, ans):
        if ans not in self.answers:
            raise ValueError('not an answer to this question')
        if self.accepted is not None:
            raise ValueError('an answer is already accepted')
        self.accepted = ans
        ans.author.reputation += REP_ACCEPTED_BONUS

    def close(self):    self.status = QuestionStatus.CLOSED
    def reopen(self):   self.status = QuestionStatus.OPEN
    def delete(self):   self.status = QuestionStatus.DELETED

class Answer(Post):
    def __init__(self, author, question, body):
        super().__init__(author, body)
        self.question = question
        question._add_answer(self)

print('core classes ready')


core classes ready


## 1. Tags + search

Real Stack Overflow lets you browse `/questions/tagged/python`. We'll add a tiny in-memory
**tag index** that keeps a reverse map `tag -> [questions]` as questions are posted.

This is just the **inverted-index** idea from search engines, scaled all the way down to a dict.


In [2]:
class QASite:
    """Holds all questions and keeps a few indexes for fast search."""
    def __init__(self):
        self.questions = []
        self._by_tag = {}

    def post_question(self, author, title, body, tags):
        q = Question(author, title, body, tags)
        self.questions.append(q)
        for t in q.tags:
            self._by_tag.setdefault(t, []).append(q)
        return q

    def search_by_tag(self, tag):
        return list(self._by_tag.get(tag, []))

    def search_text(self, keyword):
        kw = keyword.lower()
        return [q for q in self.questions
                if kw in q.title.lower() or kw in q.body.lower()]

site = QASite()
ada   = User('u1','Ada')
grace = User('u2','Grace')
site.post_question(ada,   'What is OOD?',        'Explain',          ['ood','design'])
site.post_question(ada,   'Python decorators?',  'What are they?',   ['python'])
site.post_question(grace, 'Design patterns 101', 'Observer example', ['design','patterns'])

print('by tag design :', [q.title for q in site.search_by_tag('design')])
print('text has python:', [q.title for q in site.search_text('python')])


by tag design : ['What is OOD?', 'Design patterns 101']
text has python: ['Python decorators?']


> 🧠 **Why a separate `QASite`?**
> Until now each `Question` was a free-standing object. A real site needs a *container* that
> knows about *all* questions so it can search, paginate, and enforce site-wide rules.
> We've split "one question" (entity) from "the whole site" (service) — classic **SRP**.


## 2. Badges via the observer pattern

> *"When an answer reaches score 10, award the author the **Nice Answer** badge."*

We don't want `Post` to know about badges (that would couple unrelated concerns).
Instead, the badge system **subscribes** to vote events and decides for itself.

This is the **Observer** design pattern: the post is the *subject*, the badge engine is an *observer*.


In [3]:
class BadgeEngine:
    """Listens to score changes and awards badges."""
    THRESHOLDS = [(10, 'Nice Answer'), (25, 'Good Answer'), (100, 'Great Answer')]

    def on_post_changed(self, post):
        if not isinstance(post, Answer):
            return
        for threshold, name in self.THRESHOLDS:
            if post.score >= threshold and name not in post.author.badges:
                post.author.badges.append(name)
                print(f'{post.author.name} earned badge: {name}')

def vote_v2(post, voter, v):
    """Same vote logic as notebook 2, but also fires observers at the end."""
    if voter.id == post.author.id:
        raise ValueError('cannot vote on your own post')
    prev = post.votes.get(voter.id)
    prev_val = prev.value if prev is not None else 0
    post.votes[voter.id] = v
    delta = v.value - prev_val
    per_up = REP_UPVOTE_ANSWER if isinstance(post, Answer) else REP_UPVOTE_QUESTION
    if delta > 0:
        post.author.reputation += per_up * delta
    elif delta < 0:
        post.author.reputation += REP_DOWNVOTE * (-delta)
    post._notify()

badges = BadgeEngine()
ada   = User('u1','Ada')
grace = User('u2','Grace')
voters = [User(f'v{i}', f'Voter{i}') for i in range(12)]

q = Question(ada, 'Great question', 'body', ['x'])
a = Answer(grace, q, 'Great answer')
a.subscribe(badges.on_post_changed)

for v in voters:
    vote_v2(a, v, VoteType.UP)

print('Grace badges:', grace.badges)
print('Grace rep   :', grace.reputation)


Grace earned badge: Nice Answer
Grace badges: ['Nice Answer']
Grace rep   : 120


### Why the observer pattern?

- `Post` stays tiny — no badge code in it.
- You can plug in **more** observers: send an email, push a notification, record an audit log.
- Adding a new rule (`Great Answer` at 100) is a one-line change in `BadgeEngine`.
- If you remove badges entirely, `Post` doesn't need to change.


## 3. Moderation — close, reopen, delete

Moderators can close or delete questions. The **best** design from notebook 2 already
has a `status` field; let's wrap a couple of guards and show a moderator workflow.


In [4]:
class Moderator:
    def __init__(self, user):
        self.user = user
    def close(self, q, reason):
        print(f'mod {self.user.name} closed "{q.title}" -- {reason}')
        q.close()
    def reopen(self, q):
        print(f'mod {self.user.name} reopened "{q.title}"')
        q.reopen()
    def delete(self, q):
        print(f'mod {self.user.name} deleted "{q.title}"')
        q.delete()

ada  = User('u1','Ada')
mod  = Moderator(User('m1','ModMel'))
q    = Question(ada, 'Low-effort question', '?', ['x'])

mod.close(q, reason='needs more detail')

try:
    Answer(User('u2','Bob'), q, 'late answer')
except ValueError as e:
    print('blocked:', e)

mod.reopen(q)
ans = Answer(User('u2','Bob'), q, 'now I can answer')
print('answer posted after reopen, id =', ans.id)


mod ModMel closed "Low-effort question" -- needs more detail
blocked: cannot answer a closed question
mod ModMel reopened "Low-effort question"
answer posted after reopen, id = 8


## 4. Thread-safe voting

Until now, if two voters hit the same post *at the exact same moment* we could lose an update
because `post.votes[...]` and `author.reputation += ...` are not atomic.

Let's reproduce the race, then fix it with a **per-post lock**.


In [5]:
# --- unsafe ---
class UnsafePost:
    def __init__(self, author):
        self.author = author
        self.votes = {}
    def upvote(self, voter_id):
        if voter_id in self.votes: return
        # simulate a context switch between the check and the mutate
        current = dict(self.votes)
        time.sleep(0.001)
        current[voter_id] = +1
        self.votes = current
        self.author.reputation += 10

ada = User('u1','Ada', reputation=0)
p   = UnsafePost(ada)
def spam_unsafe(vid):
    for _ in range(50):
        p.upvote(vid)
threads = [threading.Thread(target=spam_unsafe, args=(f'v{i}',)) for i in range(8)]
for t in threads: t.start()
for t in threads: t.join()
print(f'unsafe: distinct voters = {len(p.votes)}   rep = {ada.reputation}')

# --- safe ---
class SafePost:
    def __init__(self, author):
        self.author = author
        self.votes = {}
        self._lock = threading.Lock()
    def upvote(self, voter_id):
        with self._lock:
            if voter_id in self.votes: return
            self.votes[voter_id] = +1
            self.author.reputation += 10

ada2 = User('u2','Ada2', reputation=0)
p2   = SafePost(ada2)
def spam_safe(vid):
    for _ in range(50):
        p2.upvote(vid)
threads = [threading.Thread(target=spam_safe, args=(f'v{i}',)) for i in range(8)]
for t in threads: t.start()
for t in threads: t.join()
print(f'safe  : distinct voters = {len(p2.votes)}   rep = {ada2.reputation}')
assert ada2.reputation == 10 * len(p2.votes), 'safe version must satisfy rep == 10 * voters'
print('safe version invariant holds')


unsafe: distinct voters = 1   rep = 80
safe  : distinct voters = 8   rep = 80
safe version invariant holds


> 🧠 **Interview tip:** in a real system, the "lock" is usually the database
> (a `UNIQUE(user_id, post_id)` constraint plus a transaction). In-process locks are fine for a
> single-node OOD exercise — but *mention* the DB version to show you know the scale-up path.


## 5. Pluggable reputation rules — the strategy pattern

Different sites may score differently. A teaching clone might want smaller numbers.
Instead of hard-coding constants, inject a **policy** object.


In [6]:
from abc import ABC, abstractmethod

class ReputationPolicy(ABC):
    @abstractmethod
    def on_vote(self, post, delta): ...
    @abstractmethod
    def on_accept(self): ...

class ClassicPolicy(ReputationPolicy):
    def on_vote(self, post, delta):
        per_up = 10 if isinstance(post, Answer) else 5
        return per_up * delta if delta > 0 else -2 * (-delta)
    def on_accept(self): return 15

class ClassroomPolicy(ReputationPolicy):
    """Gentler rewards -- no down-vote penalty, smaller accept bonus."""
    def on_vote(self, post, delta):
        return 1 * delta if delta > 0 else 0
    def on_accept(self): return 3

def vote_v3(post, voter, v, policy):
    if voter.id == post.author.id:
        raise ValueError('cannot vote on your own post')
    prev = post.votes.get(voter.id)
    prev_val = prev.value if prev is not None else 0
    post.votes[voter.id] = v
    delta = v.value - prev_val
    post.author.reputation += policy.on_vote(post, delta)

ada   = User('u1','Ada')
grace = User('u2','Grace')
bob   = User('u3','Bob')
q = Question(ada, 'T','b',['x'])
a = Answer(grace, q, '...')

for policy_name, policy in [('classic', ClassicPolicy()), ('classroom', ClassroomPolicy())]:
    grace.reputation = 0
    # each policy starts from a fresh post so earlier votes don't leak through
    a2 = Answer(grace, q, '...')
    vote_v3(a2, bob, VoteType.UP, policy)
    print(f'{policy_name:9s}: grace rep after 1 up-vote = {grace.reputation}')


classic  : grace rep after 1 up-vote = 10
classroom: grace rep after 1 up-vote = 1


### Why this is worth doing

- The core classes never change when the rules change.
- Unit tests can pass in a `FixedRewardPolicy` (e.g., always returns `42`) to isolate the logic under test.
- Adding a brand-new rule set (monthly leaderboards?) is one new subclass.

---

## 🎁 Further extensions (try on your own)

- **Bounty** — a user spends reputation to promote a question; the bounty is transferred to the accepted answer's author.
- **Notifications** — add an `Observer` that emails the question author when a new answer is posted.
- **Search ranking** — sort by `score`, `created_at`, or a blended formula.
- **Duplicate detection** — find questions with >3 shared tags and similar titles.
- **Persistence** — swap the in-memory `QASite` for a DB-backed repository without touching any entity class.

> 🏁 **You've now walked the full OOD arc for Stack Overflow:** design → implement (bad→good→best) → extend.
> The same pattern — *clarify, model nouns/verbs, iterate, plug in patterns* — works for every other
> lab in `07-object-oriented-design/`.
